# ML-08 — Honest Model for Refresh Opportunity Scoring

**Lane locked:** Refresh / Content Opportunity Scoring.

This notebook asks whether a learned ranking can prioritize observed decline cases better than the frozen Week-4 rule. It is a decision-support exercise on the public starter snapshot, not evidence that an edit causes recovery or that the model describes Google's algorithm.

> The repository skills used for this task are `training-honest-models` and `flyrank/flyrank-data`.

## 1. Method choice and why

The operational task is **ranking**: place a limited number of review candidates at the top of a content team's queue. I use a Random Forest classifier's probability of the observed decline proxy as the ranking score.

A Random Forest fits this lane because age, staleness, exposure, CTR, position, and engagement can interact nonlinearly. I constrain depth and leaf size so the model cannot simply memorize individual rows, fix the random seed, and use permutation importance to inspect what it relies on. Complexity only earns its place if it beats the frozen rule on the same held-out rows.

**Target/proxy:** `1` when `trend_direction == "down"`. This is used only as the outcome. `trend_direction`, `trend_pct`, all last/previous-30-day inputs, product flags, providers/models, and identifiers are excluded from features.

**Primary metric:** Precision@50, because the decision is which 50 pages to review first. Precision@10, Precision@100, ROC AUC, and average precision provide supporting context.

In [1]:
from pathlib import Path
import json
import platform

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42

# Find the repository root in Colab or a local clone.
repo_root = Path.cwd()
while not (repo_root / "data" / "raw" / "content_refresh_anonymized.csv").exists():
    if repo_root == repo_root.parent:
        raise FileNotFoundError("Could not locate the starter dataset.")
    repo_root = repo_root.parent

df = pd.read_csv(repo_root / "data" / "raw" / "content_refresh_anonymized.csv")
df["is_declining_proxy"] = df["trend_direction"].eq("down").astype("int8")

# Freeze the exact Week-4 eligibility rule and score.
eligible = (
    df["impressions_90d"].ge(300)
    & df["avg_position"].gt(0)
    & df["avg_position"].le(20)
    & df["ctr"].lt(0.50)
)
model_frame = df.loc[eligible].copy()
model_frame["baseline_score"] = (
    np.log1p(model_frame["impressions_90d"])
    * (0.50 - model_frame["ctr"])
    * ((21 - model_frame["avg_position"]) / 20)
)

numeric_features = [
    "search_volume", "competition", "cpc", "word_count",
    "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "sessions_90d",
    "days_with_impressions", "days_with_sessions",
    "ctr", "avg_position", "engagement_rate",
]
categorical_features = [
    "content_type", "main_intent", "competition_level", "freshness_tier",
]
feature_columns = numeric_features + categorical_features
target = "is_declining_proxy"

print("Starter rows:", f"{len(df):,}")
print("Frozen Week-4 candidate rows:", f"{len(model_frame):,}")
print("Candidate base rate:", f"{model_frame[target].mean():.1%}")
print("Feature count before one-hot encoding:", len(feature_columns))
print("Random seed:", SEED)

Starter rows: 30,000
Frozen Week-4 candidate rows: 10,730
Candidate base rate: 63.5%
Feature count before one-hot encoding: 18
Random seed: 42


## 2. Split design

I use one reproducible **client-grouped holdout**: 75% of the 32 pseudonymized clients for training and 25% for testing. No client appears in both sets. This is stricter than a random row split because pages from the same client can share content strategy, measurement patterns, and performance norms.

Both the Week-4 rule and the model rank the **same frozen eligible rows from the same held-out clients**, and both are scored against the same proxy. The split cannot create a true future test because the starter data is one trailing-90-day snapshot; the result measures generalization to unseen clients, not future causal impact. The capstone still needs non-overlapping warehouse feature and outcome windows.

In [2]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
train_idx, test_idx = next(
    splitter.split(model_frame, groups=model_frame["client_id"])
)
train = model_frame.iloc[train_idx].copy()
test = model_frame.iloc[test_idx].copy()

train_clients = set(train["client_id"])
test_clients = set(test["client_id"])

assert train_clients.isdisjoint(test_clients)
assert len(train) + len(test) == len(model_frame)

split_summary = pd.DataFrame({
    "split": ["train", "test"],
    "rows": [len(train), len(test)],
    "pseudonymized_clients": [len(train_clients), len(test_clients)],
    "observed_decline_share": [
        train[target].mean(), test[target].mean()
    ],
})
split_summary["observed_decline_share"] = split_summary["observed_decline_share"].round(3)
display(split_summary)
print("Client overlap:", len(train_clients & test_clients))
print("Test cohort is the shared comparison set for baseline and model.")

,split,rows,pseudonymized_clients,observed_decline_share
0,train,9029,21,0.651
1,test,1701,7,0.554


Client overlap: 0
Test cohort is the shared comparison set for baseline and model.


## 3. Train and compare with my baseline

The preprocessing is learned from training rows only. Numeric missing values receive a training-set median plus a missingness indicator; categories receive the most frequent training value and one-hot encoding. The model uses balanced class weights, 400 trees, maximum depth 8, and minimum leaf size 20.

The Week-4 baseline is recomputed here from its frozen formula. Nothing is tuned on the test set. Higher scores rank first for both methods.

In [3]:
numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

model = RandomForestClassifier(
    n_estimators=400,
    max_depth=8,
    min_samples_leaf=20,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=SEED,
    n_jobs=-1,
)
pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", model),
])
pipeline.fit(train[feature_columns], train[target])

test["model_score"] = pipeline.predict_proba(test[feature_columns])[:, 1]

def precision_at_k(frame, score_column, k):
    ranked = frame.sort_values(
        [score_column, "content_id"],
        ascending=[False, True],
        kind="stable",
    )
    return float(ranked.head(k)[target].mean())

comparison = pd.DataFrame([
    {
        "method": "Base rate (no ranking)",
        "precision_at_10": test[target].mean(),
        "precision_at_50": test[target].mean(),
        "precision_at_100": test[target].mean(),
        "roc_auc": 0.50,
        "average_precision": test[target].mean(),
    },
    {
        "method": "Week-4 fixed rule",
        "precision_at_10": precision_at_k(test, "baseline_score", 10),
        "precision_at_50": precision_at_k(test, "baseline_score", 50),
        "precision_at_100": precision_at_k(test, "baseline_score", 100),
        "roc_auc": roc_auc_score(test[target], test["baseline_score"]),
        "average_precision": average_precision_score(test[target], test["baseline_score"]),
    },
    {
        "method": "Random Forest",
        "precision_at_10": precision_at_k(test, "model_score", 10),
        "precision_at_50": precision_at_k(test, "model_score", 50),
        "precision_at_100": precision_at_k(test, "model_score", 100),
        "roc_auc": roc_auc_score(test[target], test["model_score"]),
        "average_precision": average_precision_score(test[target], test["model_score"]),
    },
])
metric_columns = [c for c in comparison.columns if c != "method"]
comparison[metric_columns] = comparison[metric_columns].round(3)

print("Same held-out rows:", f"{len(test):,}")
display(comparison)

baseline_p50 = comparison.loc[
    comparison["method"].eq("Week-4 fixed rule"), "precision_at_50"
].iloc[0]
model_p50 = comparison.loc[
    comparison["method"].eq("Random Forest"), "precision_at_50"
].iloc[0]
print(
    f"Observed Precision@50 difference: {model_p50 - baseline_p50:+.1%} "
    "(model minus baseline)."
)

Same held-out rows: 1,701


,method,precision_at_10,precision_at_50,precision_at_100,roc_auc,average_precision
0,Base rate (no ranking),0.554,0.554,0.554,0.500,0.554
1,Week-4 fixed rule,0.500,0.740,0.760,0.554,0.610
2,Random Forest,1.000,0.780,0.800,0.639,0.680


Observed Precision@50 difference: +4.0% (model minus baseline).


## 4. Errors and interpretation

Permutation importance shuffles one original feature at a time on the held-out set and reports the change in ROC AUC. This is association with the model's held-out ranking—not causal feature importance.

For error reading, I call a high-ranked non-decline a **false positive for prioritization**. It may still deserve human review; it simply did not match this observed proxy. The tables below show where those errors concentrate and three concrete pseudonymized examples.

In [4]:
perm = permutation_importance(
    pipeline,
    test[feature_columns],
    test[target],
    scoring="roc_auc",
    n_repeats=8,
    random_state=SEED,
    n_jobs=-1,
)
importance = (
    pd.DataFrame({
        "feature": feature_columns,
        "importance_mean_auc_drop": perm.importances_mean,
        "importance_std": perm.importances_std,
    })
    .sort_values("importance_mean_auc_drop", ascending=False)
    .reset_index(drop=True)
)
print("Top permutation importances on held-out clients:")
display(importance.head(10).round(4))

ranked_test = test.sort_values(
    ["model_score", "content_id"], ascending=[False, True], kind="stable"
).reset_index(drop=True)
ranked_test["model_rank"] = np.arange(1, len(ranked_test) + 1)
ranked_test["position_band"] = pd.cut(
    ranked_test["avg_position"],
    bins=[0, 3, 10, 20],
    labels=["top 3", "4-10", "11-20"],
    include_lowest=True,
)
top100 = ranked_test.head(100).copy()
error_by_position = (
    top100.groupby("position_band", observed=True)
    .agg(
        n=("content_id", "size"),
        observed_decline_share=(target, "mean"),
        false_positive_count=(target, lambda s: int((s == 0).sum())),
    )
    .reset_index()
)
error_by_position["observed_decline_share"] = (
    error_by_position["observed_decline_share"].round(3)
)
print("Top-100 model errors by measured position band:")
display(error_by_position)

wrong_cases = ranked_test.loc[ranked_test[target].eq(0)].head(3).copy()
wrong_cases["why_hard"] = [
    "Strong exposure and activity resemble declining cases, but this page's observed trend proxy is not down.",
    "Its interaction of staleness, CTR, and position looks risky to the model, yet the aggregate outcome differs.",
    "The model sees a familiar client-independent pattern, but missing query and seasonality context may reverse the call.",
]
print("Three high-confidence wrong cases (pseudonymized):")
display(wrong_cases[[
    "model_rank", "content_id", "model_score", "impressions_90d",
    "ctr", "avg_position", "days_since_last_update", "why_hard",
]].round({"model_score": 3, "ctr": 3, "avg_position": 1}))

print("Interpretation:")
print(
    "The model's top features are plausible observable activity/context signals, "
    "but they do not prove that changing a page will change its outcome."
)
print(
    "Errors remain because this snapshot lacks query mix, SERP features, seasonality, "
    "and a non-overlapping future outcome window."
)

Top permutation importances on held-out clients:


,feature,importance_mean_auc_drop,importance_std
0,avg_position,0.0304,0.0030
1,days_with_sessions,0.0151,0.0019
2,sessions_90d,0.0143,0.0025
3,clicks_90d,0.0138,0.0024
4,ctr,0.0110,0.0024
5,days_with_impressions,0.0102,0.0020
6,content_age_days,0.0076,0.0085
7,impressions_90d,0.0059,0.0015
8,cpc,0.0027,0.0016
9,engagement_rate,0.0026,0.0019


Top-100 model errors by measured position band:


,position_band,n,observed_decline_share,false_positive_count
0,top 3,6,1.000,0
1,4-10,61,0.770,14
2,11-20,33,0.818,6


Three high-confidence wrong cases (pseudonymized):


,model_rank,content_id,model_score,impressions_90d,ctr,avg_position,days_since_last_update,why_hard
10,11,content_b8fe98256cb9,0.717,1623,0.18,7.6,104,Strong exposure and activity resemble declinin...
18,19,content_d788e2c18ae8,0.702,1377,0.07,11.4,104,"Its interaction of staleness, CTR, and positio..."
21,22,content_502b28c930ff,0.700,866,0.00,6.1,20,The model sees a familiar client-independent p...


Interpretation:
The model's top features are plausible observable activity/context signals, but they do not prove that changing a page will change its outcome.
Errors remain because this snapshot lacks query mix, SERP features, seasonality, and a non-overlapping future outcome window.


### What the comparison means

On this fixed grouped holdout, the model improves the primary Precision@50 over the Week-4 rule, while the full table preserves places where gains may differ by queue depth. That earns the model a place as a candidate ranking aid, not an automatic editing system.

The most important failure mode is context the snapshot cannot see: a low-click or stale-looking page may serve a branded, zero-click, seasonal, or already-changed query mix. A reviewer should inspect query-level evidence and the current page before acting.

The result is **observed and directional**. A warehouse version should define features strictly before a future outcome window, retain client-grouped or time-aware validation, and keep June 2026 sealed until the final test.

In [5]:
forbidden_features = {
    "content_id", "client_id", "trend_direction", "trend_pct",
    "is_declining_proxy",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "provider_used", "model_used",
}
assert set(feature_columns).isdisjoint(forbidden_features)
assert train_clients.isdisjoint(test_clients)
assert len(test) >= 100
assert comparison["method"].tolist() == [
    "Base rate (no ranking)", "Week-4 fixed rule", "Random Forest"
]
assert not comparison[metric_columns].isna().any().any()
assert len(wrong_cases) == 3

metrics_receipt = {
    "lane": "Refresh / Content Opportunity Scoring",
    "target_proxy": "trend_direction == down (outcome only)",
    "split": "75/25 GroupShuffleSplit by pseudonymized client_id",
    "random_seed": SEED,
    "train_rows": int(len(train)),
    "test_rows": int(len(test)),
    "train_clients": int(len(train_clients)),
    "test_clients": int(len(test_clients)),
    "primary_metric": "precision_at_50",
    "comparison": comparison.to_dict(orient="records"),
    "top_features_by_permutation_auc_drop": importance.head(5)["feature"].tolist(),
    "feature_columns": feature_columns,
    "forbidden_feature_overlap": sorted(set(feature_columns) & forbidden_features),
    "python_version": platform.python_version(),
    "sklearn_version": sklearn.__version__,
    "limitation": (
        "One trailing-window snapshot; grouped client holdout is not a "
        "non-overlapping future validation."
    ),
}
receipt_path = repo_root / "work" / "outputs" / "w05_model_metrics.json"
receipt_path.parent.mkdir(parents=True, exist_ok=True)
receipt_path.write_text(json.dumps(metrics_receipt, indent=2) + "\n")

print("Forbidden/label-derived feature overlap:", metrics_receipt["forbidden_feature_overlap"])
print("Python:", metrics_receipt["python_version"])
print("scikit-learn:", metrics_receipt["sklearn_version"])
print("Wrote:", receipt_path.relative_to(repo_root))
print("Validation: PASS")

Forbidden/label-derived feature overlap: []
Python: 3.9.6
scikit-learn: 1.6.1
Wrote: work/outputs/w05_model_metrics.json
Validation: PASS


## 5. Self-check

- [x] The method matches a ranked review decision and its choice is explained.
- [x] The test set holds out entire pseudonymized clients; there is no client overlap.
- [x] Baseline and model use the same candidate cohort, split, target, and metrics.
- [x] The comparison table includes base rate, Week-4 rule, and model.
- [x] Precision@10, Precision@50, Precision@100, ROC AUC, and average precision are visible.
- [x] Permutation importance and three concrete wrong cases are shown.
- [x] `trend_direction`, `trend_pct`, recent comparison windows, IDs, and product/provider fields are excluded from features.
- [x] Random seeds and library versions are recorded.
- [x] Claims are observed, directional, and decision-support only.
- [x] No client names, domains, URLs, private queries, credentials, or raw exports appear.
- [x] The notebook runs top to bottom with visible outputs.